# Flatand Graph Data Import Pipeline

## Approach Sketch
To be implemented...
- one node per outgoing edge, label with where it comes from
- look at switches: at fixed distance which 2 are closest -> suggest using GeoPandas API for this.
- add actions as edge attributes (LFR disjointly to all outgoing)
- export enhanced osmnx 
- write importer for enhanced osmnx to GraphRailEnv -> describe what we expect as input, which attributes etc.
- timetable for graph env
- unit test with policy reaching target (possibly hard-coded actions)
- snaphost-visualization (preferrably one osmnx export with railway features -> could be a moving house or roundabout if there is no train feature...)


## Documentation
### OSMnx and GeoPandas API documentation
* https://osmnx.readthedocs.io/en/stable/user-reference.html
* https://github.com/gboeing/osmnx-examples
* https://geopandas.org/en/stable/docs/user_guide.html

### OSM Railway documentation
* https://wiki.openstreetmap.org/wiki/Railways
* https://wiki.openstreetmap.org/wiki/Railway_stations
* https://wiki.openstreetmap.org/wiki/Tag:railway=signal
* https://wiki.openstreetmap.org/wiki/Tag:railway=switch
* https://wiki.openstreetmap.org/wiki/Tag:railway=rail
* https://wiki.openstreetmap.org/wiki/Tag:route%3Drailway


### OpenRailwayMap -> inspect Railway features
* https://openrailwaymap.app/#view=18.86/47.1194931/9.3132586&style=signals

### Insepct OSM by Node ID / Way ID
* https://www.openstreetmap.org/node/2048248546/history/4
* https://www.openstreetmap.org/way/194299504

In [ ]:
# all dependencies of this notebook, its modules and tests: see requirements.txt
# !python -m pip install -r requirements.txt
import osmnx as ox

ox.settings.log_console = True
ox.__version__

In [ ]:
# ox.settings.overpass_url='http://localhost:12347'
# ox.settings.overpass_rate_limit=False

In [ ]:
place = "Walenstadt"
# radius in metres around the geocoded place, None: within the place's boundary polygon
dist = None
# tiles = "cartodbpositron"
mk = {"radius": 6}
# cols = ["height", "addr:housenumber", "addr:street", "addr:postcode"]

## Download OSMnx graph

`ox.settings.all_oneway` controls how OSM ways become directed edges (osmnx 2.1.1, `graph.py`):

| | Default (`False`) | `all_oneway=True` |
|---|---|---|
| Ways not tagged `oneway` (most railway tracks) | Added in both directions, `(u,v)` and `(v,u)`. The reverse copy gets `reversed=True`. | Added once, in the OSM way's node order. |
| Ways with `oneway=-1` | Added once, reversed. | Added once, reversed. |
| `oneway` edge attribute | Overwritten with a bool `True`/`False`. | Not set by osmnx: the original OSM string stays, or is missing if the way has none. |

osmnx documents `all_oneway=True` as meant only for exporting via `save_graph_xml`. `osm_to_track_graph.download_rail_graph` sets it anyway (and restores it afterwards), so every way becomes a single directed edge in OSM node order.

In [ ]:
from osm_to_track_graph import download_rail_graph

G = download_rail_graph(place, dist=dist)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

## Switch-switch edges: inspect the recovery of one discarded intermediate node

The recovery is implemented in `osm_to_track_graph.py` (`download_rail_graph_fixed`); the cells below only inspect it.

`simplify_graph` collapses every chain of interstitial (non-intersection) nodes
into a single edge between the two real intersections/dead-ends it connects. If
two switches end up directly adjacent in the simplified graph, any interstitial
nodes that originally separated them (degree-2 vertices just shaping the curve)
were silently discarded - we only kept their coordinates in the edge's `geometry`.
Keeping one of them back was introduced for the earlier GGG construction attempts (removed, see the git history),
which could not represent some direct switch-to-switch adjacencies; the side-based construction below decides each
node's sides locally.

In [ ]:
from osm_to_track_graph import true_switches

# a node's OSM railway=switch tag can lag behind the actual topology (see the
# switch_attr_low_degree check further below) - a true switch is one
# with in+out degree > 2, so use that instead of the raw tag
switches_in_G = true_switches(G)
switch_switch_edges = [
    (u, v) for u, v in G.edges()
    if u in switches_in_G and v in switches_in_G
]
print(f"found {len(switch_switch_edges)} switch-switch edge(s) in the simplified graph:")
for u, v in switch_switch_edges:
    print(f"  {u} -> {v}")


In [ ]:
# same download parameters as above, but unsimplified, so we can see which
# interstitial nodes simplify_graph removed between any two switches
G_non_simple = download_rail_graph(place, dist=dist, simplify=False)


In [ ]:
# reuse osmnx's own endpoint/path-building logic (see simplify_graph's source)
# to recover exactly the interstitial path it would collapse between each pair
# of nodes in the unsimplified graph, and check it against our switch-switch edges
from osmnx.simplification import _get_paths_to_simplify

paths_in_non_simple = {
    (path[0], path[-1]): path
    for path in _get_paths_to_simplify(G_non_simple, None, None)
}

for u, v in switch_switch_edges:
    path = paths_in_non_simple.get((u, v)) or paths_in_non_simple.get((v, u))
    if path is None:
        # _get_paths_to_simplify() only yields a path when the edge needs
        # simplifying - if both endpoints are already (true, degree>2)
        # endpoints with nothing interstitial between them, it's a genuinely
        # direct edge in the raw, non-simplified data too
        assert G_non_simple.has_edge(u, v) or G_non_simple.has_edge(v, u), (
            f"edge {u}-{v} not found at all in G_non_simple"
        )
        print(f"{u} -> {v}: 0 intermediate nodes - already directly adjacent in the non-simplified graph too")
        continue
    intermediate = path[1:-1]
    print(f"{u} -> {v}: {len(intermediate)} intermediate node(s) in G_non_simple: {intermediate}")


In [ ]:
# download the unsimplified graph again and simplify it like osmnx.simplification.simplify_graph, except keeping
# one interstitial node on every switch-switch path (see osm_to_track_graph.simplify_graph_keep_switch_intermediate);
# pass filepath= to also save it as GraphML
from osm_to_track_graph import download_rail_graph_fixed

G = download_rail_graph_fixed(place, dist=dist)

switches_in_G = true_switches(G)
fixed_switch_switch_edges = [
    (u, v) for u, v in G.edges()
    if u in switches_in_G and v in switches_in_G
]
print(f"switch-switch edges remaining after the fix: {fixed_switch_switch_edges}")


In [ ]:
gdf = ox.features.features_from_place(place, 
                                      tags={"building": True}
                                     )
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)

In [ ]:
edges.explore(m=m, color="purple")

In [ ]:
nodes.explore(marker_kwds={"radius": 8})

In [ ]:
list(G.nodes(data=True))

## Visualize networkx graph

In [ ]:
import networkx as nx
nx.draw(G, labels={n[0]: str(n[0]) for n in list(G.nodes(data=True))})

## Stations

In [ ]:
stations = ox.features.features_from_place(place, tags={"railway": "station"})
stations.explore(m=m, 
                 #tiles=tiles, 
                 color="yellow", tooltip="name", marker_kwds=mk)

## Signals

In [ ]:
signals = ox.features.features_from_place(place, tags={"railway": "signal"})

In [ ]:
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)
nodes.explore(m=m, color="purple")

## Export graph to graphml

In [ ]:
ox.io.save_graphml(G, filepath="walensee.graphml")

## Import graph from graphml

In [ ]:
# loads the MultiDiGraph saved above, verifies it (no parallel edges, no interstitial nodes etc. - see
# load_track_graph's docstring) and returns it as a DiGraph
from track_graph import load_track_graph

GG = load_track_graph("walensee.graphml")

## Graph Rail Env

In [ ]:
for n, d in GG.nodes(data=True):
    if "railway" not in d or d["railway"] != "switch":
        continue
    print(type(n))
    print(n,d)

In [ ]:
SWITCH=2042222936

In [ ]:
for n in list(GG.successors(SWITCH)):
    print(n, GG.get_edge_data(SWITCH, n, default=None))

In [ ]:
TARGET=2042193124

In [ ]:
for n in list(GG.predecessors(SWITCH)):
    print(n, GG.get_edge_data(n, SWITCH, default=None))

In [ ]:
from flatland.core.env_observation_builder import DummyObservationBuilder
from flatland.envs.graph.rail_graph_transition_map import GraphTransitionMap
from flatland.core.graph.graph_resource_map import GraphResourceMap
from flatland.envs.graph.distance_map import GraphDistanceMap
from flatland.envs.graph_rail_env import GraphRailEnv
from flatland.envs.timetable_utils import Timetable
from flatland.envs import malfunction_generators as mal_gen
from flatland.envs.rewards import DefaultRewards
from flatland.envs.rail_env import RailEnvActions

In [ ]:
# add actions: all rail env actions on every edge - GG isn't switch-aware (a node's outgoing edges follow the
# OSM ways' drawing direction, not the switch's point/legs), so its actions are not partitioned at switches and dead
# ends; see "add actions to GGG" below for the partitioned actions
for v1,v2, edge_data in GG.edges(data=True):
    edge_data["actions"] = set(RailEnvActions)
    #GG.edges(v1,v2)["actions"]= {RailEnvActions.MOVE_FORWARD}
    # print(v1,v2,edge_data)

In [ ]:
resource_map = {}
for n1, n2 in GG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[SWITCH], [TARGET]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    latest_arrivals=[[None,5]],
    max_episode_steps=99,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=DummyObservationBuilder(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.reset()

In [ ]:
GG.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.rail.g.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.reset()
for _ in range(99):
    print("==============================")
    print(f"elapsed before step {graph_env._elapsed_steps}")
    for agent in graph_env.agents:
        print(f"agent.initial_entry_point={agent.initial_entry_point}")
        print(f"agent.current_entry_point={agent.current_entry_point}")
        print(f"agent.targets={agent.targets}")
        print(f"agent.speed_counter.speed={agent.speed_counter.speed}")
        print(f"agent.speed_counter.max_speed={agent.speed_counter.max_speed}")
        print(f"agent.state={agent.state}")
        
    graph_env.step({0: RailEnvActions.MOVE_FORWARD})
    if graph_env.dones["__all__"]:
        print("==============================")
        print("done")
        break

## Switch geometry: which two legs stay closest?

For the switch node `SWITCH` (from the osmnx graph `GG` loaded back from `walensee.graphml`) we walk
`DISTANCE` metres along each of its three incident edges (starting at the switch) and compare the
resulting probe points pairwise: the pair that stays closest together is the one that continues
through the switch in (nearly) the same direction, the remaining leg is the odd one out.

In [ ]:
import itertools

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point

from switch_geometry import DISTANCE, oriented_geometry


# GG was downcast to a plain DiGraph above (for the flatland env), but osmnx's
# projection/plotting helpers need a MultiDiGraph - rebuild one just for this section
GG_multi = nx.MultiDiGraph(GG)

# project to UTM so that lengths and interpolation are in metres
G_proj = ox.projection.project_graph(GG_multi)

# all edges incident to the switch, no matter their direction in the digraph
incident = ([(SWITCH, v, k, v) for _, v, k in G_proj.out_edges(SWITCH, keys=True)]
            + [(u, SWITCH, k, u) for u, _, k in G_proj.in_edges(SWITCH, keys=True)])

probes = {}
for u, v, k, neighbour in incident:
    geom = oriented_geometry(G_proj, u, v, k, SWITCH)
    if geom.length < DISTANCE:
        print(f"edge {SWITCH} -> {neighbour} is only {geom.length:.1f} m long, "
              f"probe clipped to its far end")
    probes[neighbour] = geom.interpolate(DISTANCE)

probe_points = gpd.GeoDataFrame({"neighbour": list(probes)},
                                geometry=list(probes.values()),
                                crs=G_proj.graph["crs"])
probe_points

In [ ]:
pairs = pd.DataFrame(
    [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
    columns=["neighbour_1", "neighbour_2", "distance"],
).sort_values("distance", ignore_index=True)
display(pairs)

closest = pairs.iloc[0]
print(f"at {DISTANCE} m from {SWITCH} the closest pair is "
      f"{closest.neighbour_1} / {closest.neighbour_2} ({closest.distance:.3f} m apart); "
      f"odd one out: {set(probes) - {closest.neighbour_1, closest.neighbour_2}}")

In [ ]:
# visual check: the switch (blue), its three incident edges (green), the three original
# neighbours (orange) and the three probe points at DISTANCE along each edge (red)
ego = nx.ego_graph(G_proj, SWITCH, radius=1, undirected=True)
ego_nodes, ego_edges = ox.convert.graph_to_gdfs(ego)
m_switch = ego_edges.to_crs(epsg=4326).explore(color="green")
# ego_nodes is indexed by osmid (not a column); explore() resets the index itself
# and surfaces it as an "osmid" tooltip column, so don't add that column again here
ego_nodes.loc[[SWITCH]].to_crs(epsg=4326).explore(m=m_switch, color="blue", tooltip="osmid", marker_kwds=mk)
ego_nodes.loc[list(probes)].to_crs(epsg=4326).explore(m=m_switch, color="orange", tooltip="osmid", marker_kwds=mk)
probe_points.to_crs(epsg=4326).explore(m=m_switch, color="red", tooltip="neighbour", marker_kwds=mk)

## Switch geometry as a reusable function (see `switch_geometry.py`)

In [ ]:
from switch_geometry import legs_from_geometry, switch_probes

neighbor_1, neighbor_2 = legs_from_geometry(G_proj, SWITCH)
print(f"closest neighbours of switch {SWITCH}: {neighbor_1} and {neighbor_2}")


## Switch geometry: forward and diverging leg

In [ ]:
from switch_geometry import switch_legs

undirected = GG.to_undirected()
for switch in [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]:
    (point,) = set(undirected.neighbors(switch)) - {int(x) for x in legs_from_geometry(G_proj, switch)}
    forward, diverging, diverges_right = switch_legs(G_proj, switch, point)
    print(f"switch {switch}: from point {point}, forward leg {forward}, "
          f"diverging leg {diverging} to the {'right' if diverges_right else 'left'}")

## Sanity check: run `legs_from_geometry` on every switch

In [ ]:
switches = [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]
print(f"found {len(switches)} node(s) with in+out degree > 2: {switches}")

missing_switch_attr = [n for n in switches if GG.nodes[n].get("railway") != "switch"]
assert not missing_switch_attr, (
    f"node(s) with in+out degree > 2 but no switch attribute: {missing_switch_attr}"
)
print("OK: all nodes with in+out degree > 2 have the switch attribute")

switch_attr_low_degree = [
    n for n, d in GG.nodes(data=True)
    if d.get("railway") == "switch" and GG.in_degree(n) + GG.out_degree(n) <= 2
]
print(f"node(s) with switch attribute but in+out degree <= 2: {switch_attr_low_degree}")

for switch in switches:
    try:
        n1, n2 = legs_from_geometry(G_proj, switch)
        print(f"switch {switch}: closest neighbours are {n1} and {n2}")
    except Exception as e:
        print(f"switch {switch}: legs_from_geometry FAILED with {type(e).__name__}: {e}")

## Visual inspection: switches vs. low-degree switch-tagged nodes

Plot the whole graph, highlighting nodes with in+out degree > 2 (the switches used above) in one
color and nodes that carry the `switch` attribute but have degree <= 2 (flagged by the sanity
check above) in another.

In [ ]:
from matplotlib.lines import Line2D

switches_set = set(switches)
switch_attr_low_degree_set = set(switch_attr_low_degree)

default_color = "#999999"
switch_color = "#1f77b4"
low_degree_switch_color = "#d62728"

node_color = [
    low_degree_switch_color if n in switch_attr_low_degree_set
    else switch_color if n in switches_set
    else default_color
    for n in GG_multi.nodes
]

fig, ax = ox.plot.plot_graph(
    GG_multi,
    node_color=node_color,
    node_size=15,
    edge_color="w",
    edge_linewidth=0.5,
    show=False,
    close=False,
)
ax.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="", color=switch_color, label="switch (in+out degree > 2)"),
        Line2D([], [], marker="o", linestyle="", color=low_degree_switch_color, label="switch attribute, in+out degree <= 2"),
        Line2D([], [], marker="o", linestyle="", color=default_color, label="other node"),
    ],
    loc="lower left",
)

In [ ]:
from track_graph import track_node


def explore_categorized(graph: nx.MultiDiGraph | nx.DiGraph) -> "folium.Map":
    """Interactive map of `graph` (GG, or GGG with vertices "x" / "x_prime"): edges green, nodes colored by the switch
    category of their GG node."""
    # ox.convert.graph_to_gdfs() needs (u, v, key) edges, i.e. a MultiDiGraph
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)
    nodes, edges = ox.convert.graph_to_gdfs(multi_graph)

    m = edges.explore(color="green")

    gg_nodes = nodes.index.map(lambda n: track_node(n) if isinstance(n, str) else n)
    is_switch = gg_nodes.isin(switches_set)
    is_low_degree_switch = gg_nodes.isin(switch_attr_low_degree_set)
    is_other = ~(is_switch | is_low_degree_switch)

    if is_other.any():
        nodes[is_other].explore(m=m, color=default_color, tooltip="osmid", marker_kwds=mk)
    if is_switch.any():
        nodes[is_switch].explore(m=m, color=switch_color, tooltip="osmid", marker_kwds=mk)
    if is_low_degree_switch.any():
        nodes[is_low_degree_switch].explore(m=m, color=low_degree_switch_color, tooltip="osmid", marker_kwds=mk)

    return m


# same categories as above, but as an interactive map (see "visual check" above)
explore_categorized(GG_multi)


## Sanity check: max degree in the graph

In [ ]:
over_degree = {n: GG.in_degree(n) + GG.out_degree(n) for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 3}
assert not over_degree, f"node(s) with in+out degree > 3: {over_degree}"
print("OK: all nodes in GG have in+out degree <= 3")

## Side-based construction: double vertex graph (`track_graph.build_double_vertex_graph`)

Every GG node `x` becomes two GGG nodes: `"x"` leaves `x` to side 0 of its tracks, `"x_prime"` to side 1 - at a switch,
its legs are on side 0 and its point on side 1. Every GG track `u - v` becomes two GGG edges, one per direction of
travel: leaving `u` to the side facing `v`, the train enters `v` from the side facing `u` and so leaves `v` to the other
side. Each node's sides are decided locally from its own tracks, nothing is propagated: the construction is
deterministic, never fails and runs in O(edges). This is a double vertex graph as introduced by M. Montigel (Formal representation of track topologies by double
vertex graphs, COMPRAIL 92). See `build_double_vertex_graph`'s docstring for its guarantees, and
`test_track_graph.py` for the tests on `walensee.graphml` and `sargans_reversing_loop.graphml`.

In [ ]:
from track_graph import build_double_vertex_graph

GGG = build_double_vertex_graph(GG)

In [ ]:
from track_graph import opposite_vertex

# edge duality, see build_double_vertex_graph's docstring - test_track_graph.py also checks that GGG has exactly the
# moves possible in GG
mismatches = [(a, b) for a, b in GGG.edges if not GGG.has_edge(opposite_vertex(b), opposite_vertex(a))]
assert not mismatches, f"edge duality violated for: {mismatches}"
print("OK: a -> b in GGG iff opposite_vertex(b) -> opposite_vertex(a)")

## Display GGG as interactive map

In [ ]:
# same categories as above (see "Visual inspection" / interactive map section)
explore_categorized(GGG)

In [ ]:
len(GGG.edges())

In [ ]:
for e in GGG.edges():
    print(e)

In [ ]:
import networkx as nx
nx.draw(GGG, labels={n[0]: str(n[0]) for n in list(GGG.nodes(data=True))})

In [ ]:
# add actions to GGG (cf. "# add actions" for GG above) such that at every node, each rail env action is either
# prohibited or in the actions of exactly one outgoing edge - see double_vertex_graph_to_env.add_actions() (create_env()
# below adds them itself, on a copy - here just to show them per switch)
from double_vertex_graph_to_env import add_actions

add_actions(GGG, GG)
for n in GGG.nodes:
    successors = list(GGG.successors(n))
    if len(successors) == 2:
        print(n, {l: GGG.edges[n, l]["actions"] for l in successors})

In [ ]:
from double_vertex_graph_to_env import create_env

# GGG nodes are str(n) / f"{n}_prime" (not GG's int nodes). Longest switch-to-switch
# shortest path in GGG: 14 edges, through 7 other switches
# (2042222936, 2042193124, 2048050594, 2048050596, 2048248467, 2048248475, 2048248546)
SWITCH = "2042222934"  # switch of the triangular junction
TARGET = "2048248489"  # switch
assert nx.has_path(GGG, SWITCH, TARGET), f"no directed path {SWITCH} -> {TARGET} in GGG"
print(nx.shortest_path(GGG, SWITCH, TARGET))

# edge lengths are osmnx's `length` (metres): the 14 edges sum to ~3760 m, which take ~3760 steps at speed 1
graph_env = create_env(GGG, GG, source=SWITCH, target=TARGET, speed=1, latest_arrival=8000, max_episode_steps=10000)

## Shortest-path policy on GGG

In [ ]:
from flatland.envs.step_utils.states import TrainState

# follows the shortest path (in number of edges) to the target
from double_vertex_graph_to_env import GraphShortestPathPolicy

In [ ]:
from pathlib import Path

from double_vertex_graph_to_env import run_shortest_path_policy

data_dir = Path("trajectories")
trajectory = run_shortest_path_policy(graph_env, data_dir / "shortest_path")
display(trajectory.trains_positions)
print(graph_env.agents[0].state, graph_env.agents[0].current_entry_point)
assert graph_env.agents[0].state == TrainState.DONE, f"train did not arrive: {graph_env.agents[0].state}"


## Step through the episode on the map

In [ ]:
# step through an episode with GraphShortestPathPolicy, showing the train on the map:
# each train is an OSM-like feature tagged historic=locomotive, drawn with that tag's
# icon from the OSM wiki - N.B. the tag is only interpreted here (client-side), the
# rendered OSM tiles know nothing about it.
# The map (ipyleaflet) is created once and only its layers are updated on each step,
# so zoom/pan are kept.
import base64
import io
import math
import shutil
import urllib.request

import ipywidgets as widgets
from ipyleaflet import CircleMarker, Icon, LayerGroup, Map, Marker, Polyline
from IPython.display import display
from PIL import Image, ImageOps

from flatland.trajectories.policy_runner import PolicyRunner
from flatland.trajectories.trajectories import Trajectory

from coordinates import position_lat_lon

# OSM tag -> icon (https://wiki.openstreetmap.org/wiki/File:Historic_locomotive.png, CC BY-SA)
TAG_ICONS = {
    ("historic", "locomotive"): "https://upload.wikimedia.org/wikipedia/commons/e/ef/Historic_locomotive.png",
}


def mercator_y(lat: float) -> float:
    """Web Mercator y of latitude `lat` (in the same unit as longitude in radians)."""
    sin = math.sin(math.radians(lat))
    return math.log((1 + sin) / (1 - sin)) / 2


def edge_orientation(g: nx.DiGraph, u: str, v: str) -> tuple[float, bool]:
    """`(rotation, mirrored)` of the icon such that its front (chimney and lamp) faces the direction of edge (u, v)
    on the map, with its rails still at the bottom - approximated by the straight vector from u to v (no
    interpolation along the edge's geometry). Rotation in degrees clockwise; `mirrored`: use the horizontally
    mirrored icon (front facing west), as rotating alone would turn the icon upside down on westbound edges."""
    dx = math.radians(g.nodes[v]["x"] - g.nodes[u]["x"])
    dy = mercator_y(g.nodes[v]["y"]) - mercator_y(g.nodes[u]["y"])
    bearing = math.degrees(math.atan2(dx, dy))  # clockwise from north
    mirrored = dx < 0
    # unrotated, the icon's front faces east (bearing 90), the mirrored one's west (bearing 270)
    return (bearing - (270 if mirrored else 90)) % 360, mirrored


def mirrored_icons(icon_url: str, **icon_kwargs) -> dict[bool, Icon]:
    """The icon at `icon_url` as is (False) and mirrored horizontally (True), both as PNG data URLs - Leaflet
    markers can only be rotated, not mirrored."""
    request = urllib.request.Request(icon_url, headers={"User-Agent": "flatland-scenarios osmnx_graph_import notebook"})
    with urllib.request.urlopen(request) as response:
        image = Image.open(io.BytesIO(response.read()))
    icons = {}
    for mirrored in (False, True):
        buffer = io.BytesIO()
        (ImageOps.mirror(image) if mirrored else image).save(buffer, format="PNG")
        icons[mirrored] = Icon(icon_url="data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode(), **icon_kwargs)
    return icons


def train_features(env: GraphRailEnv) -> list[dict]:
    """One OSM-like feature (tags + location + rotation along its edge) per train, location None if not on the map."""
    return [
        {
            "handle": agent.handle,
            "tags": {"historic": "locomotive", "name": f"train {agent.handle}"},
            "location": position_lat_lon(env, agent.handle),
            "orientation": edge_orientation(env.rail.g, c, n) if (c := agent.current_entry_point) is not None and (n := agent.next_entry_point) is not None else (0, False),
            "title": f"train {agent.handle}: {c} ({agent.state.name})",
        }
        for agent in env.agents
    ]


def edge_locations(g: nx.DiGraph, u: str, v: str) -> list[tuple[float, float]]:
    """(lat, lon) points along edge (u, v) - its geometry if any, else the straight segment."""
    geometry = g.edges[u, v].get("geometry")
    if geometry is not None:
        return [(y, x) for x, y in geometry.coords]
    return [(g.nodes[n]["y"], g.nodes[n]["x"]) for n in (u, v)]


def create_map(g: nx.DiGraph) -> Map:
    """Static layers of GGG: edges green, nodes colored by switch category (cf.
    explore_categorized()) - plain/_prime twins share their geometry, draw them once."""
    track = {frozenset((track_node(u), track_node(v))): (u, v) for u, v in g.edges}
    nodes = {track_node(n): n for n in g.nodes}
    # fixed size, so that fit_to_bounds() can compute the zoom level fitting the graph
    m = Map(scroll_wheel_zoom=True, layout=widgets.Layout(width=f"{MAP_WIDTH}px", height=f"{MAP_HEIGHT}px"))
    m.add(Polyline(locations=[edge_locations(g, u, v) for u, v in track.values()], color="green", fill=False, weight=3))
    m.add(LayerGroup(layers=[
        CircleMarker(
            location=(g.nodes[n]["y"], g.nodes[n]["x"]), radius=mk["radius"], weight=1, fill_opacity=0.8,
            color=(low_degree_switch_color if osmid in switch_attr_low_degree_set
                   else switch_color if osmid in switches_set else default_color),
        )
        for osmid, n in nodes.items()
    ]))
    return m


def graph_bounds(g: nx.DiGraph) -> list[list[float]]:
    """[[south, west], [north, east]] of all nodes of `g`."""
    lats = [g.nodes[n]["y"] for n in g.nodes]
    lons = [g.nodes[n]["x"] for n in g.nodes]
    return [[min(lats), min(lons)], [max(lats), max(lons)]]


# N.B. not Map.fit_bounds(): it zooms out/in step by step comparing against Map.bounds, which the rendered map
# reports asynchronously - stale bounds make it zoom out far too much (e.g. to all of Europe). Instead, compute the
# largest Web Mercator zoom level at which `bounds` fit into the map's fixed pixel size (256px tiles).
MAP_WIDTH, MAP_HEIGHT = 800, 600


def fit_to_bounds(m: Map, bounds: list[list[float]]):
    """Center `m` on `bounds` ([[south, west], [north, east]]) and zoom in as far as they still fit."""
    (south, west), (north, east) = bounds
    lat_fraction = (mercator_y(north) - mercator_y(south)) / (2 * math.pi)
    lon_fraction = (east - west) / 360
    min_zoom = 0 if m.min_zoom is None else m.min_zoom
    max_zoom = 18 if m.max_zoom is None else m.max_zoom
    zooms = [math.log2(MAP_HEIGHT / 256 / lat_fraction) if lat_fraction > 0 else max_zoom,
             math.log2(MAP_WIDTH / 256 / lon_fraction) if lon_fraction > 0 else max_zoom]
    m.center = ((south + north) / 2, (west + east) / 2)
    m.zoom = max(min_zoom, min(max_zoom, math.floor(min(zooms))))


# for visualization, step through the episode on a copy of GGG with all edge lengths divided by the shortest
# edge's length (so the shortest edge has length 1) - with lengths in metres, it would take thousands of steps
shortest_edge_length = min(length for _, _, length in GGG.edges(data="length"))
GGG_step = GGG.copy()
for _, _, edge_data in GGG_step.edges(data=True):
    edge_data["length"] = edge_data["length"] / shortest_edge_length
step_env = create_env(GGG_step, GG, source=SWITCH, target=TARGET, speed=1, latest_arrival=8000, max_episode_steps=10000)

# start from an empty dir (see above)
shutil.rmtree(data_dir / "step", ignore_errors=True)
step_env.reset()
step_policy = GraphShortestPathPolicy()
step_runner = PolicyRunner(
    policy=step_policy,
    trajectory=Trajectory.create_empty(data_dir=data_dir / "step", env=None),
    by_pass_env=step_env,
)

step_map = create_map(step_env.rail.g)
# remaining shortest path per train, and the trains themselves - updated in place by show()
path_layers = {agent.handle: Polyline(locations=[], color="orange", fill=False, weight=6, opacity=0.8) for agent in step_env.agents}
train_markers = {}
train_icons = {}
for feature in train_features(step_env):
    icon_url = next(url for (k, v), url in TAG_ICONS.items() if feature["tags"].get(k) == v)
    train_icons[feature["handle"]] = mirrored_icons(icon_url, icon_size=[30, 30], icon_anchor=[15, 15])
    train_markers[feature["handle"]] = Marker(
        location=feature["location"] or step_map.center, visible=False, draggable=False,
        icon=train_icons[feature["handle"]][False],
        rotation_origin="center center",
    )
for layer in [*path_layers.values(), *train_markers.values()]:
    step_map.add(layer)

step_button = widgets.Button(description="Step", icon="step-forward")
# number of env steps executed per click
steps_per_click = widgets.BoundedIntText(value=1, min=1, max=10_000, description="steps", layout=widgets.Layout(width="150px"))
step_label = widgets.Label()


def show(fit_bounds: bool = False):
    """Update label, remaining paths and train markers; with `fit_bounds`, also zoom the map to the graph."""
    agent = step_env.agents[0]
    speed_counter = agent.speed_counter
    if speed_counter.distance is None:
        position = "off map"
    else:
        position = (f"on {agent.current_entry_point} -> {agent.next_entry_point}, "
                    f"offset {float(speed_counter.distance):.2f}/{float(speed_counter.edge_length):.2f}, "
                    f"speed {float(speed_counter.speed):.2f} "
                    # one length unit of GGG_step is the shortest edge's length in metres
                    f"({float(speed_counter.speed) * shortest_edge_length:.2f} m per step)")
    step_label.value = f"step {step_env._elapsed_steps}: {agent.state.name} {position}"
    for handle, path_layer in path_layers.items():
        # the policy only drops a train's path on its next act_many() after DONE
        path = [] if step_env.agents[handle].state == TrainState.DONE else step_policy._shortest_paths.get(handle, [])
        path_layer.locations = [edge_locations(step_env.rail.g, u, v) for u, v in zip(path, path[1:])]
    for feature in train_features(step_env):
        marker = train_markers[feature["handle"]]
        marker.visible = feature["location"] is not None
        if feature["location"] is not None:
            marker.location = feature["location"]
            rotation, mirrored = feature["orientation"]
            icon = train_icons[feature["handle"]][mirrored]
            if marker.icon is not icon:
                marker.icon = icon
            marker.rotation_angle = rotation
        marker.title = feature["title"]
    if fit_bounds:
        fit_to_bounds(step_map, graph_bounds(step_env.rail.g))


def on_step(_):
    done = False
    for _ in range(steps_per_click.value):
        _, done = step_runner.step()
        if done:
            break
    step_button.disabled = done
    show()


step_button.on_click(on_step)
display(widgets.VBox([widgets.HBox([step_button, steps_per_click, step_label]), step_map]))
# fit only initially - on_step()'s show() keeps the user's zoom/pan
show(fit_bounds=True)
